[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllInVaders/aistudio-full-course/blob/main/notebooks/01_Setup_Models_and_Token_Economics.ipynb)

# Module 01: Google AI Studio Setup, Model Family Discovery & Token Economics
### Módulo 01: Configuración de Google AI Studio, Familias de Modelos y Economía de Tokens

**English Overview**: Configure the official unified `google-genai` SDK, verify your `GEMINI_API_KEY` (or Vertex AI ADC), probe the current model families — **Gemini 3.8 Flash**, **Nano Banana**, **Gemini Omni Flash**, and **Gemini 3.8 Live** — and benchmark preflight token accounting with `client.models.count_tokens`. You will also run your first call on the **Interactions API**, the standard surface used throughout this course.

**Resumen en Español**: Configura el SDK unificado oficial `google-genai`, verifica tu `GEMINI_API_KEY` (o credenciales ADC de Vertex AI), explora las familias de modelos actuales — **Gemini 3.8 Flash**, **Nano Banana**, **Gemini Omni Flash** y **Gemini 3.8 Live** — y mide el consumo de tokens con `client.models.count_tokens`. También ejecutarás tu primera llamada a la **Interactions API**, la superficie estándar que usaremos en todo el curso.

Referencias: [catálogo de modelos](https://ai.google.dev/gemini-api/docs/models) · [Interactions API](https://ai.google.dev/gemini-api/docs/interactions-overview)

In [ ]:
# Step 1: Install the official unified Google GenAI SDK
%pip install -q -U google-genai pydantic

In [ ]:
# Step 2: Configure your API Key (works in Google Colab or local Jupyter)
import os
from google import genai
from google.genai import types

try:
    from google.colab import userdata  # type: ignore
    os.environ.setdefault('GEMINI_API_KEY', userdata.get('GEMINI_API_KEY'))
except Exception:
    pass

client = genai.Client()

# The default workhorse for every lab in this course.
DEFAULT_MODEL = 'gemini-3.8-flash'
print('Initialized google-genai client successfully!')

## 1. Probing the Current Model Families / Descubrimiento de Modelos

Rather than bucketing the whole catalog by loose substring matches, we check the
five specific models this course depends on. That turns `client.models.list()`
into an actionable readiness report for your key or project.

| Model ID | Family | Used for |
| :--- | :--- | :--- |
| `gemini-3.8-flash` | Gemini 3.8 Flash | Text, reasoning, structured output, agents |
| `gemini-3.1-flash-image` | Nano Banana 2 | Default production image generation |
| `gemini-3-pro-image` | Nano Banana Pro | 4K hero renders, precise in-image text |
| `gemini-omni-1.1-flash` | Gemini Omni Flash | Video generation and editing |
| `gemini-3.8-live` | Gemini 3.8 Live | Real-time voice and video agents |

In [ ]:
PROBE_TARGETS = [
    (DEFAULT_MODEL, 'Gemini 3.8 Flash', 'Text, reasoning, structured output, agents'),
    ('gemini-3.1-flash-image', 'Nano Banana 2', 'Default production image generation'),
    ('gemini-3-pro-image', 'Nano Banana Pro', '4K hero renders, precise in-image text'),
    ('gemini-omni-1.1-flash', 'Gemini Omni Flash', 'Video generation and editing'),
    ('gemini-3.8-live', 'Gemini 3.8 Live', 'Real-time voice and video agents'),
]

available = {(m.name or '').replace('models/', '').lower() for m in client.models.list()}

for model_id, label, purpose in PROBE_TARGETS:
    if model_id in available:
        status = 'AVAILABLE'
    elif any(name.startswith(model_id) for name in available):
        status = 'AVAILABLE (aliased)'
    else:
        status = 'NOT VISIBLE'
    print(f'{model_id:<24} {status:<20} {label} - {purpose}')

## 2. Preflight Token Counting & Cost Budgeting / Conteo de Tokens y Presupuesto

Before sending large multimodal prompts in production, call
`client.models.count_tokens()` to verify input token counts and enforce
per-request budget ceilings.

The cell below uses `client.models.generate_content`, the **classic compatible
path**. It is still fully supported and is the easiest way to read
`usage_metadata`, which is exactly what we want for cost accounting.

In [ ]:
prompt = (
    'You are an AI Product Studio strategist. Summarize the 3 pillars of a '
    'high-converting product launch brief in English and Spanish.'
)
token_info = client.models.count_tokens(model=DEFAULT_MODEL, contents=prompt)
print('Preflight token count:', token_info.total_tokens)

# Classic / compatible path - best for inspecting usage_metadata.
response = client.models.generate_content(
    model=DEFAULT_MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=300),
)
print('\nUsage Metadata:', response.usage_metadata)
print('\nModel Response:\n', response.text)

## 3. Your First Interactions API Call / Tu Primera Llamada a la Interactions API

`client.interactions.create` is the modern, stateful surface, and it is what
image and video generation now require. Every remaining module in this course
uses it.

Two things to notice:

1. The prompt goes in `input=`, not `contents=`, and the reply comes back on
   `interaction.output_text`.
2. Reasoning depth is controlled by `generation_config={'thinking_level': ...}`,
   which accepts `'low'`, `'medium'`, or `'high'`.

> **Legacy migration note.** Gemini 3 replaced the numeric `thinking_budget`
> parameter with `thinking_level`. If you are porting older code, see
> https://ai.google.dev/gemini-api/docs/migrate-to-interactions

In [ ]:
interaction = client.interactions.create(
    model=DEFAULT_MODEL,
    input=prompt,
    generation_config={'thinking_level': 'low'},
)

print('Interaction ID:', getattr(interaction, 'id', '(not returned)'))
print('\nModel Response:\n', interaction.output_text)

# Stateful follow-up: no need to resend the conversation history.
followup = client.interactions.create(
    model=DEFAULT_MODEL,
    input='Now compress that into a single 20-word elevator pitch.',
    previous_interaction_id=interaction.id,
    generation_config={'thinking_level': 'low'},
)
print('\nFollow-up:\n', followup.output_text)

### Retention: the `store` parameter / Retención: el parámetro `store`

Interactions are **stored server-side by default** (`store=true`). Retention is
**55 days on the Paid Tier** and **1 day on the Free Tier**. That storage is what
makes `previous_interaction_id` work.

Set `store=False` to opt out entirely. Note the trade-off:

| | `store=True` (default) | `store=False` |
| :--- | :--- | :--- |
| Server-side retention | 55 days paid / 1 day free | none |
| `previous_interaction_id` | works | disabled |
| `background=true` | compatible | incompatible |

Use `store=False` for one-shot calls that handle untrusted or sensitive input;
keep the default whenever you need multi-turn or conversational editing.

Docs: https://ai.google.dev/gemini-api/docs/interactions-overview

In [ ]:
# One-shot call with retention disabled. Because nothing is stored, this
# interaction's id cannot be used as a previous_interaction_id later.
private = client.interactions.create(
    model=DEFAULT_MODEL,
    input='Summarize the three pillars of a product launch brief.',
    store=False,
    generation_config={'thinking_level': 'low'},
)
print(private.output_text)